# Predicting asset failure within 90 days

**Question:** Can inspection-time information predict `failed_90d`?

This notebook builds a reproducible, leakage-aware baseline. The primary evaluation is a chronological holdout: models train on the earliest 80% of inspections and are tested on the newest 20%. This is stricter and more realistic than a random split when deployment predicts future inspections.

Key rules:

- `asset_tag` is excluded because it is an identifier and every value is unique.
- `parts_replaced` is excluded from the deployable model because the dictionary says it is recorded on the work order that *closed* the inspection record. It is also nearly a proxy for the target, so using it would give an implausibly optimistic estimate.
- Calendar fields derived from `inspected_at` are allowed, but their behavior is inspected for temporal drift.
- Ranking is assessed with ROC AUC and average precision (PR AUC). Probability quality is assessed with Brier score. Because failures are uncommon, accuracy is not a useful headline metric.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:
DATA_PATH = Path("data.csv")
DICTIONARY_PATH = Path("columns.csv")

data = pd.read_csv(DATA_PATH, parse_dates=["inspected_at"])
dictionary = pd.read_csv(DICTIONARY_PATH)

required = set(dictionary["column"])
assert required == set(data.columns), "Data columns do not match the dictionary."
assert data["failed_90d"].dropna().isin([0, 1]).all(), "Target must be binary."

display(dictionary)
display(data.head())


## Data audit

In [ ]:
audit = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing": data.isna().sum(),
    "unique": data.nunique(dropna=False),
})

summary = pd.Series({
    "rows": len(data),
    "duplicate rows": int(data.duplicated().sum()),
    "unique assets": data["asset_tag"].nunique(),
    "start date": data["inspected_at"].min(),
    "end date": data["inspected_at"].max(),
    "failures": int(data["failed_90d"].sum()),
    "failure rate": data["failed_90d"].mean(),
}, name="value")

display(summary.to_frame())
display(audit)


In [ ]:
monthly = (
    data.assign(month=data["inspected_at"].dt.to_period("M").dt.to_timestamp())
        .groupby("month")["failed_90d"]
        .agg(inspections="size", failures="sum", failure_rate="mean")
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=data, x="failed_90d", ax=axes[0], color="#4C78A8")
axes[0].set(title="Target balance", xlabel="Failed within 90 days", ylabel="Rows")
monthly["failure_rate"].plot(ax=axes[1], marker="o", color="#E45756")
axes[1].axhline(data["failed_90d"].mean(), ls="--", color="gray", label="overall")
axes[1].set(title="Failure rate changes substantially over time", ylabel="Failure rate", xlabel="Inspection month")
axes[1].legend()
plt.tight_layout()
display(monthly)


The target prevalence is not stable over time. This matters twice: it makes a random split overly reassuring, and it means even a model with useful ranking may produce poorly calibrated probabilities in a later period. Before production, confirm that every inspection has a complete 90-day outcome window; otherwise recent zeros may be right-censored rather than genuine non-failures.

## Leakage check

In [ ]:
parts_target = pd.crosstab(
    data["parts_replaced"], data["failed_90d"], margins=True
)
display(parts_target)

print("Failure rate when parts_replaced = 0:",
      f"{data.loc[data.parts_replaced.eq(0), 'failed_90d'].mean():.3%}")
print("Failure rate when parts_replaced >= 2:",
      f"{data.loc[data.parts_replaced.ge(2), 'failed_90d'].mean():.3%}")


`parts_replaced` separates the classes almost perfectly: there are no failures when it is 0 and no non-failures when it is 2 or more. Combined with its definition as a value on the work order that closes the inspection, this is a strong leakage warning. The primary model therefore excludes it. A later sensitivity analysis quantifies how misleading the result would be if it were included.

## Feature engineering and chronological validation

In [ ]:
model_data = data.sort_values("inspected_at").reset_index(drop=True).copy()
model_data["inspection_month"] = model_data["inspected_at"].dt.month.astype(str)
model_data["inspection_year"] = model_data["inspected_at"].dt.year.astype(str)

numeric_features = [
    "vibration_rms", "bearing_temp_c", "ambient_temp_c", "meter_hours", "load_pct"
]
categorical_features = ["site", "inspection_month", "inspection_year"]
features = numeric_features + categorical_features
target = "failed_90d"

split_idx = int(len(model_data) * 0.80)
train = model_data.iloc[:split_idx].copy()
test = model_data.iloc[split_idx:].copy()

split_summary = pd.DataFrame({
    "rows": [len(train), len(test)],
    "start": [train.inspected_at.min(), test.inspected_at.min()],
    "end": [train.inspected_at.max(), test.inspected_at.max()],
    "failures": [train[target].sum(), test[target].sum()],
    "failure_rate": [train[target].mean(), test[target].mean()],
}, index=["train", "test"])
display(split_summary)


In [ ]:
preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

models = {
    "Dummy prevalence": DummyClassifier(strategy="prior"),
    "Logistic regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=2_000)),
    ]),
    "Random forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=10,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
}

# Expanding-window validation only within the training period.
tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []
for model_name, estimator in models.items():
    if model_name == "Dummy prevalence":
        continue
    for fold, (fit_idx, val_idx) in enumerate(tscv.split(train), start=1):
        fitted = clone(estimator).fit(train.iloc[fit_idx][features], train.iloc[fit_idx][target])
        prob = fitted.predict_proba(train.iloc[val_idx][features])[:, 1]
        y_val = train.iloc[val_idx][target]
        cv_rows.append({
            "model": model_name,
            "fold": fold,
            "validation_prevalence": y_val.mean(),
            "roc_auc": roc_auc_score(y_val, prob),
            "average_precision": average_precision_score(y_val, prob),
        })

cv_results = pd.DataFrame(cv_rows)
display(cv_results)
display(cv_results.groupby("model")[["roc_auc", "average_precision"]].agg(["mean", "std"]))


## Final evaluation on the newest 20%

In [ ]:
def top_fraction_metrics(y_true, probability, fraction=0.10):
    n = max(1, int(len(y_true) * fraction))
    chosen = np.argsort(probability)[-n:]
    y = np.asarray(y_true)
    precision = y[chosen].mean()
    recall = y[chosen].sum() / y.sum()
    return precision, recall, precision / y.mean()


test_rows = []
fitted_models = {}
test_probabilities = {}

for model_name, estimator in models.items():
    fitted = clone(estimator).fit(train[features], train[target])
    probability = fitted.predict_proba(test[features])[:, 1]
    top_precision, top_recall, top_lift = top_fraction_metrics(test[target], probability)
    test_rows.append({
        "model": model_name,
        "roc_auc": roc_auc_score(test[target], probability),
        "average_precision": average_precision_score(test[target], probability),
        "brier_score": brier_score_loss(test[target], probability),
        "mean_predicted_risk": probability.mean(),
        "top_10pct_precision": top_precision,
        "top_10pct_recall": top_recall,
        "top_10pct_lift": top_lift,
    })
    fitted_models[model_name] = fitted
    test_probabilities[model_name] = probability

test_results = pd.DataFrame(test_rows).set_index("model")
display(test_results.style.format("{:.3f}"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for model_name, probability in test_probabilities.items():
    fpr, tpr, _ = roc_curve(test[target], probability)
    precision, recall, _ = precision_recall_curve(test[target], probability)
    axes[0].plot(fpr, tpr, label=f"{model_name} ({test_results.loc[model_name, 'roc_auc']:.3f})")
    axes[1].plot(recall, precision, label=f"{model_name} ({test_results.loc[model_name, 'average_precision']:.3f})")

axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(title="ROC curve — chronological holdout", xlabel="False-positive rate", ylabel="True-positive rate")
axes[1].axhline(test[target].mean(), ls="--", color="gray", label="holdout prevalence")
axes[1].set(title="Precision–recall curve — chronological holdout", xlabel="Recall", ylabel="Precision")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()


In [ ]:
# Bootstrap uncertainty for the strongest ranking model.
best_name = test_results["average_precision"].idxmax()
best_probability = test_probabilities[best_name]
y_test = test[target].to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
bootstrap = []
for _ in range(2_000):
    idx = rng.integers(0, len(y_test), len(y_test))
    if np.unique(y_test[idx]).size < 2:
        continue
    bootstrap.append([
        roc_auc_score(y_test[idx], best_probability[idx]),
        average_precision_score(y_test[idx], best_probability[idx]),
    ])
bootstrap = np.asarray(bootstrap)
intervals = pd.DataFrame({
    "estimate": [test_results.loc[best_name, "roc_auc"], test_results.loc[best_name, "average_precision"]],
    "95% CI lower": np.quantile(bootstrap, 0.025, axis=0),
    "95% CI upper": np.quantile(bootstrap, 0.975, axis=0),
}, index=["ROC AUC", "Average precision"])
display(Markdown(f"**Best holdout ranking model: {best_name}**"))
display(intervals.style.format("{:.3f}"))


## What drives the model?

In [ ]:
# Permuting raw columns preserves an interpretable importance at the source-feature level.
importance = permutation_importance(
    fitted_models[best_name], test[features], test[target],
    scoring="roc_auc", n_repeats=30, random_state=RANDOM_STATE, n_jobs=-1,
)
importance_table = (
    pd.DataFrame({
        "feature": features,
        "importance_mean": importance.importances_mean,
        "importance_sd": importance.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)
display(importance_table)

plt.figure(figsize=(8, 4))
sns.barplot(data=importance_table, x="importance_mean", y="feature", color="#4C78A8")
plt.axvline(0, color="black", lw=1)
plt.title("Permutation importance on chronological holdout (ROC AUC decrease)")
plt.xlabel("Mean decrease in ROC AUC")
plt.ylabel("")
plt.tight_layout()


## Leakage sensitivity analysis

In [ ]:
leaky_numeric = numeric_features + ["parts_replaced"]
leaky_features = features + ["parts_replaced"]
leaky_preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), leaky_numeric),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])
leaky_model = Pipeline([
    ("preprocess", leaky_preprocessor),
    ("model", LogisticRegression(max_iter=2_000)),
])
leaky_model.fit(train[leaky_features], train[target])
leaky_probability = leaky_model.predict_proba(test[leaky_features])[:, 1]

leakage_comparison = pd.DataFrame({
    "ROC AUC": [test_results.loc["Logistic regression", "roc_auc"], roc_auc_score(test[target], leaky_probability)],
    "Average precision": [test_results.loc["Logistic regression", "average_precision"], average_precision_score(test[target], leaky_probability)],
    "Brier score": [test_results.loc["Logistic regression", "brier_score"], brier_score_loss(test[target], leaky_probability)],
}, index=["Deployable features", "Including parts_replaced (leaky)"])
display(leakage_comparison.style.format("{:.3f}"))


## Conclusion

In [ ]:
best = test_results.loc[best_name]
baseline_ap = test[target].mean()
top_n = int(len(test) * 0.10)
top_failures = int(round(best["top_10pct_recall"] * test[target].sum()))

display(Markdown(f'''**Yes, but only as a risk-ranking model at this stage.** On the newest {len(test):,} inspections ({int(test[target].sum())} failures), the best model achieved ROC AUC **{best['roc_auc']:.3f}** and average precision **{best['average_precision']:.3f}**, versus a no-skill average precision of **{baseline_ap:.3f}**. The highest-risk 10% ({top_n} inspections) contained **{top_failures} of {int(test[target].sum())} failures**, a **{best['top_10pct_lift']:.1f}× lift** over inspecting assets at random.

That is useful signal for prioritization, but not yet enough evidence for automatic maintenance decisions:

1. Only 31 failures occur in the chronological holdout, so uncertainty is material.
2. Failure prevalence shifts sharply over time; the best model's mean predicted risk is **{best['mean_predicted_risk']:.1%}** while observed holdout prevalence is **{baseline_ap:.1%}**, so probability calibration is not stable.
3. Confirm that all rows—especially recent inspections—had a complete 90-day follow-up window. Retrain on mature labels only if not.
4. Keep `parts_replaced` out unless the prediction is explicitly made after that value becomes available and its operational meaning is verified. Its near-perfect separation is a leakage red flag.
5. Validate on a later, fully matured time window and choose an operating threshold from the cost of missed failures versus extra inspections. Repeated inspections per asset, failure mode, asset class/age, maintenance history, and prior sensor trends would likely improve performance.
'''))
